In [16]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [21]:
df = spark.read.csv(
    "week5_dataset.csv",
    header=True,
    inferSchema=True
)

df.show(5)

+-------+----------------+------+----------------+-----------+---------+---------+---+------------+-------------------+-----------------+--------+-----+--------+
|user_id|transaction_date|region|product_category|sale_amount|   status|     city|age|subscription|      raw_timestamp|            email|username|price|store_id|
+-------+----------------+------+----------------+-----------+---------+---------+---+------------+-------------------+-----------------+--------+-----+--------+
|    101|      2026-06-01|  West|     Electronics|        400|     NULL|    Delhi| 18|     Premium|2026-06-01 10:00:00|user101@gmail.com| user101|  400|      S1|
|    102|      2026-06-02|  East|        Clothing|        500|Completed|   Mumbai| 19|       Basic|2026-06-02 10:00:00|user102@gmail.com| user102|  500|      S2|
|    103|      2026-06-03| North|         Grocery|        600|  Pending|Bangalore| 20|     Premium|2026-06-03 10:00:00|user103@gmail.com| user103|  600|      S3|
|    104|      2026-06-04| S

In [23]:
q3 = df.dropDuplicates(["user_id", "transaction_date"])

print("Original Rows:", df.count())
print("After Removing Duplicates:", q3.count())

q3.show()

Original Rows: 60
After Removing Duplicates: 50
+-------+----------------+------+----------------+-----------+---------+---------+---+------------+-------------------+-----------------+--------+-----+--------+
|user_id|transaction_date|region|product_category|sale_amount|   status|     city|age|subscription|      raw_timestamp|            email|username|price|store_id|
+-------+----------------+------+----------------+-----------+---------+---------+---+------------+-------------------+-----------------+--------+-----+--------+
|    101|      2026-06-01|  West|     Electronics|        400|     NULL|    Delhi| 18|     Premium|2026-06-01 10:00:00|user101@gmail.com| user101|  400|      S1|
|    102|      2026-06-02|  East|        Clothing|        500|Completed|   Mumbai| 19|       Basic|2026-06-02 10:00:00|user102@gmail.com| user102|  500|      S2|
|    103|      2026-06-03| North|         Grocery|        600|  Pending|Bangalore| 20|     Premium|2026-06-03 10:00:00|user103@gmail.com| user

In [24]:
q4 = df.filter(col("region") == "West") \
       .groupBy("product_category") \
       .agg(avg("sale_amount").alias("avg_sale_amount"))

q4.show()

+----------------+------------------+
|product_category|   avg_sale_amount|
+----------------+------------------+
|         Grocery|            1040.0|
|     Electronics|1066.6666666666667|
|        Clothing|            1040.0|
+----------------+------------------+



In [25]:
q5 = df.na.fill({"status": "Unknown"})

q5.select("status").show()

+---------+
|   status|
+---------+
|  Unknown|
|Completed|
|  Pending|
|  Unknown|
|Completed|
|  Pending|
|  Unknown|
|Completed|
|  Pending|
|  Unknown|
|Completed|
|  Pending|
|  Unknown|
|Completed|
|  Pending|
|  Unknown|
|Completed|
|  Pending|
|  Unknown|
|Completed|
+---------+
only showing top 20 rows


In [29]:
df_large = df

for i in range(4):
    df_large = df_large.union(df_large)

print(df_large.count())new_df = df.drop("price")

960


In [30]:
q6 = df_large.groupBy("city") \
             .agg(count("*").alias("city_count")) \
             .filter(col("city_count") > 100)

q6.show()

+---------+----------+
|     city|city_count|
+---------+----------+
|Bangalore|       192|
|  Chennai|       192|
|   Mumbai|       192|
|     Pune|       192|
|    Delhi|       192|
+---------+----------+



In [34]:
new_df = df.drop("price")
df.printSchema()
new_df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- store_id: string (nullable = true)

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: integer (nullable = true)
 |-- status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- raw_timestamp: timestamp (nullable = true

In [35]:
q8 = df.filter(
    (col("age") >= 18) &
    (col("age") <= 30) &
    (col("subscription") == "Premium")
)

q8.show()

+-------+----------------+------+----------------+-----------+---------+---------+---+------------+-------------------+-----------------+--------+-----+--------+
|user_id|transaction_date|region|product_category|sale_amount|   status|     city|age|subscription|      raw_timestamp|            email|username|price|store_id|
+-------+----------------+------+----------------+-----------+---------+---------+---+------------+-------------------+-----------------+--------+-----+--------+
|    101|      2026-06-01|  West|     Electronics|        400|     NULL|    Delhi| 18|     Premium|2026-06-01 10:00:00|user101@gmail.com| user101|  400|      S1|
|    103|      2026-06-03| North|         Grocery|        600|  Pending|Bangalore| 20|     Premium|2026-06-03 10:00:00|user103@gmail.com| user103|  600|      S3|
|    105|      2026-06-05|  West|        Clothing|        800|Completed|     Pune| 22|     Premium|2026-06-05 10:00:00|user105@gmail.com| user105|  800|      S2|
|    107|      2026-06-07| N

In [36]:
q10 = df.withColumn(
    "event_time",
    col("raw_timestamp").cast(TimestampType())
).drop("raw_timestamp")

q10.show()

+-------+----------------+------+----------------+-----------+---------+---------+---+------------+-----------------+--------+-----+--------+-------------------+
|user_id|transaction_date|region|product_category|sale_amount|   status|     city|age|subscription|            email|username|price|store_id|         event_time|
+-------+----------------+------+----------------+-----------+---------+---------+---+------------+-----------------+--------+-----+--------+-------------------+
|    101|      2026-06-01|  West|     Electronics|        400|     NULL|    Delhi| 18|     Premium|user101@gmail.com| user101|  400|      S1|2026-06-01 10:00:00|
|    102|      2026-06-02|  East|        Clothing|        500|Completed|   Mumbai| 19|       Basic|user102@gmail.com| user102|  500|      S2|2026-06-02 10:00:00|
|    103|      2026-06-03| North|         Grocery|        600|  Pending|Bangalore| 20|     Premium|user103@gmail.com| user103|  600|      S3|2026-06-03 10:00:00|
|    104|      2026-06-04| S

In [37]:
q12 = df.filter(
    col("email").isNotNull() &
    (col("username") != "")
)

q12.show()

+-------+----------------+------+----------------+-----------+---------+---------+---+------------+-------------------+-----------------+--------+-----+--------+
|user_id|transaction_date|region|product_category|sale_amount|   status|     city|age|subscription|      raw_timestamp|            email|username|price|store_id|
+-------+----------------+------+----------------+-----------+---------+---------+---+------------+-------------------+-----------------+--------+-----+--------+
|    101|      2026-06-01|  West|     Electronics|        400|     NULL|    Delhi| 18|     Premium|2026-06-01 10:00:00|user101@gmail.com| user101|  400|      S1|
|    102|      2026-06-02|  East|        Clothing|        500|Completed|   Mumbai| 19|       Basic|2026-06-02 10:00:00|user102@gmail.com| user102|  500|      S2|
|    103|      2026-06-03| North|         Grocery|        600|  Pending|Bangalore| 20|     Premium|2026-06-03 10:00:00|user103@gmail.com| user103|  600|      S3|
|    105|      2026-06-05|  

In [38]:
q13 = df.agg(
    min("price").alias("min_price"),
    max("price").alias("max_price"),
    avg("price").alias("avg_price")
)

q13.show()

+---------+---------+------------------+
|min_price|max_price|         avg_price|
+---------+---------+------------------+
|      400|     2300|1183.3333333333333|
+---------+---------+------------------+



In [39]:
q15 = df.dropDuplicates() \
        .na.fill({"price": 0}) \
        .groupBy("store_id") \
        .agg(sum("price").alias("total_revenue"))

q15.show()

+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|      S3|        19600|
|      S2|        21300|
|      S1|        21600|
+--------+-------------+

